In [ ]:
import tiktoken
import torch
import gc
import time

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

tokenizer = tiktoken.get_encoding("gpt2")
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Shortened context length (orig: 1024)
    "emb_dim": 1600,  # Embedding dimension
    "n_heads": 25,  # Number of attention heads
    "n_layers": 48,  # Number of layers
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": True,  # Query-key-value bias
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from SLM.GPT import GPTModel
from SLM.LoadModel import load_weights_into_gpt

model = GPTModel()
load_weights_into_gpt(model, "Xl")
model.half()
model = model.to(device)
model.eval()

In [ ]:
import json

file_path = "summaryWithContext.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)
data = data[:500]
print("Number of entries:", len(data))

In [2]:
import re


def clean_text(text):
    """
    Cleans the input text by removing unwanted patterns, special characters, and normalizing whitespace.
    """

    # Remove pronunciation guides (e.g., /ˈkɑːmələ ˈdeɪvi/)
    text = re.sub(r"\/.*?\/|\(.*?\)", "", text)

    # Remove special characters (e.g., ;, -, etc.)
    # text = re.sub(r'[;,\-()]', ' ', text)

    # Remove square brackets shit
    text = re.sub(r"\[.*?\]", "", text)

    # Normalize whitespace
    # text = re.sub(r'\s+', ' ', text).strip()

    # Remove Unicode characters (e.g., \u02c8)
    text = re.sub(r"\\u[0-9a-fA-F]{4}", "", text)

    # Remove dates (e.g., October 20, 1964)
    # text = re.sub(r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b', '', text)

    # Remove titles (e.g., Dr., Mr., Ms., etc.)
    text = re.sub(r"\b(?:Dr|Mr|Ms|Mrs|Prof)\.\s*", "", text)

    # Remove newlines (\n) and replace with a space
    # text = re.sub(r'\n', ' ', text)

    # Normalize whitespace again
    # text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# import json
# with open('summary.json') as f:
#     data = json.load(f)

# cleaned_data = []

# # Process each entry in the data
# for entry in data:
#     if isinstance(entry['response'], str) and entry['context'] != '' and len(entry['context'].split()) < 30 and len(entry['response'].split()) < 30:
#         # Clean the context
#         cleaned_context = clean_text(entry['context'])
#         cleaned_response = clean_text(entry['response'])
#         cleaned_question = clean_text(entry['question'])
#         # Create a summary dictionary
#         summary = {
#             'question': cleaned_question,
#             'context': cleaned_context,
#             'response': cleaned_response
#         }

#         # Add the summary to the cleaned data list
#         cleaned_data.append(summary)

# # Save the cleaned and filtered data to summaryWithContext.json
# with open('summaryWithContext.json', mode='w') as f:
#     f.write(json.dumps(cleaned_data, indent=2))

# print("Data cleaned and saved to summaryWithContext.json")

Data cleaned and saved to summaryWithContext.json


In [ ]:
# with open('timetable.json', "r", encoding="utf-8") as file:
#         timetable = json.load(file)
# course_codes = {}
# for key, value in timetable['faculty_details'].items():
#     course_codes[key] = value['course_title']

# def format_input(entry):
#     instruction_text = (
#         f"Given the query about today's schedule and the provided context, "
#         f"generate a response that clearly conveys the scheduled activity in a natural and concise manner."
#         f"\n\n### Query:\n{entry['query']}\n"
#     )

#     isfirst = True
#     day = entry["current_time"].split(" ")[-1]
#     todays_schedule = timetable["time_table"][day]
#     now = datetime.now()
#     today9am = now.replace(hour=9, minute=0, second=0, microsecond=0)
#     today4pm = now.replace(hour=16, minute=0, second=0, microsecond=0)
#     today430pm = now.replace(hour=16, minute=20, second=0, microsecond=0)

#     time_table_dict = {}
#     for key in todays_schedule:
#         text = todays_schedule[key].replace("/","").split(" ")
#         text = [x for x in text if x.strip()]
#         start_hour, end_hour, am_pm = key[:2], key[3:5], key[6:]
#         time_table_time = datetime.strptime(f"{start_hour} {am_pm}", "%I %p").replace(year=now.year, month=now.month, day=now.day)
#         if(len(text) > 0):
#             input_text = f"{text[0]} of {text[2].replace("C:","")} in Room {text[4]}\n"
#         else:
#             if(isfirst and time_table_time > today9am):
#                 input_text = f"Lunch break"
#                 isfirst = False
#             else:
#                 input_text = f"No class right now"
#         time_table_dict[f"{time_table_time.strftime('%H:%M')}"] = input_text

#     result = "pp"
#     curr_time = datetime.strptime(entry["current_time"], '%H:%M %A')
#     if(curr_time.replace(year=now.year, month=now.month, day=now.day) > today4pm):
#         if(curr_time.replace(year=now.year, month=now.month, day=now.day) > today430pm):
#             result = "Classes are over for today."
#         else:
#             result = time_table_dict[f"16:00"]
#     else:
#         for time in time_table_dict:
#             time_table_time = (datetime.strptime(time, '%H:%M') + timedelta(minutes=7))
#             if(curr_time < time_table_time):
#                 lecture = time_table_dict[time].strip()
#                 if(lecture == "No class right now" and time != f"16:00"):
#                     continue
#                 else:
#                     if(lecture == "No class right now"):
#                         result = f"{lecture}"
#                     else:
#                         result = f"{lecture} at {time} today."
#                     break
#     for key, value in course_codes.items():
#         if(key in result):
#             result = result.replace(key, value)
#             break
#     input_text = f"\n### Context:\n" + result
#     return instruction_text + input_text

In [ ]:
# def format_input(entry):
#     instruction_text = (
#         f"Describe the weather based on the given temperature."
#     )

#     input_text = f"\n\n### Context:\n{entry['context']}" if entry["context"] else ""

#     return instruction_text + input_text

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is a question and a context. "
        f"Your task is to generate an accurate response based on the provided context."
        f"\n\n### Question:\n{entry['question']}"
    )

    input_text = f"\n\n### Context:\n{entry['context']}" if entry["context"] else ""

    return instruction_text + input_text

In [ ]:
instruction_plus_input = format_input(data[10])
response_text = f"\n\n### Answer:\n{data[10]['response']}"
full_text = instruction_plus_input + response_text
print(full_text)

In [ ]:
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{str(entry['response'])}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
def custom_collate_fn(
    batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item) + 1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn, device=device, allowed_max_length=1024
)

In [ ]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)  # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 1

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate(
    model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        idx_cond = idx_cond.to(device)
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(device), logits
            )

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if (
            idx_next == eos_id
        ):  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx.to(device), idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    model = model.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()  # Helps prevent NaN gradients


def train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    tokenizer,
):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    global_step = 0
    gc.collect()
    torch.cuda.empty_cache()
    # Main training loop
    count = 5
    model = model.to(device)

    for epoch in range(num_epochs):
        gc.collect()
        torch.cuda.empty_cache()
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            with autocast():
                logits = model(input_batch)
                loss = torch.nn.functional.cross_entropy(
                    logits.flatten(0, 1), target_batch.flatten()
                )

            scaler.scale(loss).backward()  # Scales gradients to prevent underflow
            scaler.step(optimizer)
            scaler.update()  # Adjusts scaling factor dynamically

            global_step += 1
            tokens_seen += input_batch.numel()
            x = 600
            # if((global_step+x) % x == 0):
            #     print(f"Saving Model Count {count}....")
            #     torch.save({
            #         "model_state_dict": model.state_dict(),
            #         "optimizer_state_dict": optimizer.state_dict(),
            #         },
            #         f"Checkpoint/checkpoint_{count}.pth"
            #     )
            #     break
            #     gc.collect()
            #     torch.cuda.empty_cache()
            #     count +=1
            #     time.sleep(120)
            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}"
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

    return train_losses, val_losses, track_tokens_seen

In [ ]:
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1, fused=True
)
num_epochs = 1


train_losses, val_losses, tokens_seen = train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    tokenizer=tokenizer,
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# entry = {
# "question": "Tell me about gojo",
# "context": "Satoru Gojo (五条悟 Gojō Satoru?) is one of the main protagonists of the Jujutsu Kaisen series. He is a special grade jujutsu sorcerer and widely recognized as the strongest in the world. Satoru is the pride of the Gojo Clan, the first person to inherit both the Limitless and the Six Eyes in four hundred years. He works as a teacher at the Tokyo Jujutsu High and uses his influence to protect and train strong young allies. "
# }

entry = {
    "question": "Who is Rama?",
    "context": "Rama (/ˈrɑːmə/;[4] Sanskrit: राम, IAST: Rāma, Sanskrit: [ˈraːmɐ] ⓘ) is a major deity in Hinduism. He is worshipped as the seventh and one of the most popular avatars of Vishnu.[5] In Rama-centric Hindu traditions, he is considered the Supreme Being. Also considered as the ideal man (maryāda puruṣottama), Rama is the male protagonist of the Hindu epic Ramayana. His birth is celebrated every year on Rama Navami, which falls on the ninth day of the bright half (Shukla Paksha) of the lunar cycle of Chaitra (March–April), the first month in the Hindu calendar.[6][7]",
}
entry["context"] = clean_text(entry["context"])
entry["question"] = clean_text(entry["question"])

input_text = format_input(entry)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer).to(device),
    max_new_tokens=256,
    context_size=1024,
    top_k=30,
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(input_text) :].strip()

# print(response_text.strip())
# print("----\n")
print(generated_text)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.show()


epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
checkpoint = torch.load(f"Checkpoint/checkpoint_4.pth")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)